# **Initialize Gemini API**

In [70]:
from google import genai
import json

client = genai.Client()
model = "gemini-3.8-flash"

def read_sas():
    try:
        with open("../data/Project_1/SAS Programs/0 - Data and Measure Standardization_2025Jul.sas", "r", encoding='utf8') as file:
            content = file.read()
            return content
    except FileNotFoundError:
        return f"Error: The file path was not found."

output = read_sas()

# **Get Completion Function**
General purpose function to make an LLM call.

Parameters:
- prompt
    - A user's query
- system_message
    - Provide's the LLM with a role
- model
    - Model to use (e.g. gemini 3.8 flash)
- client
    - gemini, openai, etc
- previous_interaction_id
    - Provide the LLM with context from a pervious call

In [71]:
def get_completion(
        prompt: str, 
        system_message: str, 
        model: str, 
        client,
        previous_interaction_id: str | None = None,
    ) -> tuple[str, str]:
    
    response = client.interactions.create(
        model=model,
        system_instruction=system_message,
        input=prompt,
        previous_interaction_id=previous_interaction_id,
    )

    if not response.output_text:
        raise RuntimeError("Model returned with no text.")

    return response.output_text, response.id

## **Translator Output Contract**

Every completed translation must be represented by one JSON object with exactly these five fields:

- `code`: the complete translated function.
- `checkpoints`: checkpoint calls used during translation.
- `conflicts`: places where the specification and SAS source disagree.
- `assumptions`: assumptions made by the Translator.
- `helpers_requested`: helper functions needed by the translation.

The four metadata fields are arrays and must be empty arrays when there is nothing to report. Responses must contain JSON only, without Markdown fences or surrounding commentary.

In [72]:
TRANSLATION_RESULT_FIELDS = (
    "code",
    "checkpoints",
    "conflicts",
    "assumptions",
    "helpers_requested",
)

TRANSLATION_OUTPUT_CONTRACT = {
    "code": "string containing the complete translated function",
    "checkpoints": "array of checkpoint calls used",
    "conflicts": "array of specification and SAS source disagreements",
    "assumptions": "array of Translator assumptions",
    "helpers_requested": "array of helper functions needed",
}


## **Translator Input Contract**

The Translator receives five inputs from the previous agent, November. The SAS source is authoritative when it disagrees with the specification; the disagreement must be reported in `conflicts`.

- `specification`: the target interface, required logic, SAS behaviors to preserve, and checks.
- `sas_source`: the source code for the unit and the authority for its behavior.
- `upstream_signatures`: signatures of upstream units and accepted helpers. Signatures are provided without code bodies.
- `behavior_notes`: cited SAS behavior notes for constructs found in the unit.
- `project_conventions`: fixed project rules, including one function per unit, no file input or output inside units, missing-value representation, the checkpoint call, and policy rules.

The Translator uses the specification to understand what to build, the SAS source to resolve behavior, the signatures to call existing functionality, the behavior notes to preserve exact SAS semantics, and the project conventions to shape the implementation.

In [73]:
TRANSLATOR_INPUT_FIELDS = (
    "specification",
    "sas_source",
    "upstream_signatures",
    "behavior_notes",
    "project_conventions",
)

TRANSLATOR_INPUT_CONTRACT = {
    "specification": "target interface, logic, SAS behaviors, and checks",
    "sas_source": "authoritative SAS source for the unit",
    "upstream_signatures": "signatures of upstream units and accepted helpers",
    "behavior_notes": "cited SAS behavior notes for constructs in the unit",
    "project_conventions": "fixed rules for unit structure, I/O, missing values, checkpoints, and policies",
}


# **Reflection Workflow**

## **Initial Translation**

**Initial Translation Function**

Parameters:
- src_code_language
    - The language of the current code
- trg_code_language
    - The language of the code you want to translate too
- src_code
    - The code itself

In [74]:
def initial_translation(
        src_code_language: str,
        trg_code_language: str,
        src_code: str
):
    system_prompt = '''You are an expert translator between programming languages.
        Return JSON only with exactly these fields: code, checkpoints, conflicts, assumptions, helpers_requested.
        code must contain the complete translated function and the other fields must be arrays.
    '''

    translation_prompt = f'''
    Translate the following {src_code_language} code to {trg_code_language}. Preserve all logic, variable names, and
    functionality.

    Return one JSON object with exactly these five fields:
    - code: the complete translated function.
    - checkpoints: checkpoint calls used during translation.
    - conflicts: disagreements between the specification and source code.
    - assumptions: decisions made where the source or requirements are ambiguous.
    - helpers_requested: helper functions needed by the translation.

    Use an empty array when a metadata field has no entries. Return JSON only, with no Markdown fences or commentary.

    [Source Code]:
    <source_code>
    {src_code}
    <source_code>
    '''

    res, id = get_completion(translation_prompt, system_prompt, model, client)

    return res, id

## **Reflection**

In [75]:
def reflection(
        src_code_language: str,
        trg_code_language: str,
        src_code: str,
        prev_inter_id: str
):
    system_prompt = f'''
        You are reviewing a {src_code_language}-to-{trg_code_language} translation.
        The source code is authoritative for behavior. Identify disagreements with the specification as conflicts,
        and identify uncertain translator decisions as assumptions or helper needs.
        Return JSON only with these fields: overall_assessment, issues, missing_checkpoints, semantic_risks,
        required_changes, unresolved_questions.
    '''

    reflection_prompt = f'''
        Carefully compare each line of the source code and the translation, then provide actionable feedback for
        the improvement call.
        
        You may assume the source code is correct and no improvements need to be made to the source code.
        
        The overall goal is to ensure that the translated code matches the source code's behavior (1:1).

        NOTE: The translated text resides within the previous interaction.
        The source code, delimited by XML tags <SOURCE_CODE></SOURCE_CODE>, are as follows:

        <SOURCE_CODE>
        {src_code}
        </SOURCE_CODE>

        Return one JSON object with exactly these fields:
        - overall_assessment: a concise summary.
        - issues: concrete translation issues.
        - missing_checkpoints: required checkpoint calls not present.
        - semantic_risks: behavior that may differ from the source.
        - required_changes: changes needed in the final result.
        - unresolved_questions: remaining ambiguities.

        Use empty arrays when a field has no entries. Return JSON only, with no Markdown fences or commentary.

        When reviewing, account for:
        - Syntax Correctness
            - Attributes and Operations
            - Dependencies
        - Logic / Behavior of the code
        - Output Produced
        - Performance; Time & Space
            - Is the translation slower or faster than the other?
            - Does the translation take more space or less space?
    '''

    res, id = get_completion(reflection_prompt, system_prompt, model, client, prev_inter_id)

    return res, id

## **Improved Translation**

In [76]:
def final_translation(
        src_code_language: str,
        trg_code_language: str,
        src_code: str,
        prev_inter_id: str
):
    system_prompt = f"""
        You are an expert in translating {src_code_language} to {trg_code_language}.
        Improve the translation using the source code and the reflection from the previous interaction.
        Return JSON only with exactly these fields: code, checkpoints, conflicts, assumptions, helpers_requested.
        The code field must contain the complete final translated function, and the other fields must be arrays.
    """

    translation_prompt = f"""
        Carefully improve the {src_code_language}-to-{trg_code_language} translation using the reflection feedback:

        /source_code/
        {src_code}
        /source_code/

        Ensure you take the criticism into account and preserve the source behavior. Return one JSON object with
        exactly these five fields:
        - code: the complete final translated function.
        - checkpoints: checkpoint calls used during translation.
        - conflicts: disagreements between the specification and source code.
        - assumptions: decisions made where the source or requirements are ambiguous.
        - helpers_requested: helper functions needed by the translation.

        Use an empty array when a metadata field has no entries. Return JSON only, with no Markdown fences or commentary.

        Look out for:
        1. Accuracy / Correctness of code
        2. Precision (Is the output identical?)
        3. Is it 1:1? (No unnecessary code or redundancy)

        This is a translation, make sure you do not add any extra details other than the entire block of code that
        has been translated.

    """

    res, id = get_completion(translation_prompt, system_prompt, model, client, prev_inter_id)

    return res, id

## **Putting It All Together**

In [77]:
def translate_with_reflection(
        src_code_language: str,
        trg_code_language: str,
        src_code: str,
        translator_input: dict | None = None,
):
    """Run translation, reflection, and improvement as one workflow."""

    workflow_source = src_code
    if translator_input:
        workflow_source = (
            f"<SAS_SOURCE>\n{src_code}\n</SAS_SOURCE>\n"
            f"<TRANSLATOR_INPUT>\n{json.dumps(translator_input)}\n</TRANSLATOR_INPUT>"
        )

    initial_res, initial_id = initial_translation(
        src_code_language,
        trg_code_language,
        workflow_source,
    )
    reflection_res, reflection_id = reflection(
        src_code_language,
        trg_code_language,
        workflow_source,
        initial_id,
    )
    final_res, _ = final_translation(
        src_code_language,
        trg_code_language,
        workflow_source,
        reflection_id,
    )

    try:
        final_result = json.loads(final_res)
    except json.JSONDecodeError as error:
        raise ValueError("Final translation was not valid JSON.") from error

    if not isinstance(final_result, dict):
        raise ValueError("Final translation must be a JSON object.")

    missing_fields = set(TRANSLATION_RESULT_FIELDS) - final_result.keys()
    extra_fields = final_result.keys() - set(TRANSLATION_RESULT_FIELDS)
    if missing_fields or extra_fields:
        raise ValueError(
            f"Final translation fields are invalid. Missing: {missing_fields}; Extra: {extra_fields}."
        )

    if not isinstance(final_result["code"], str) or not final_result["code"].strip():
        raise ValueError("Final translation code must be a non-empty string.")

    metadata_fields = TRANSLATION_RESULT_FIELDS[1:]
    if any(not isinstance(final_result[field], list) for field in metadata_fields):
        raise ValueError("Final translation metadata fields must be arrays.")

    return final_result
 

## **Sample Test Case**

In [78]:
sample_source = "def add_one(value):\n    return value + 1"
sample_artifact = translate_with_reflection("Python", "Java", sample_source)
print(json.dumps(sample_artifact, indent=2))


{
  "code": "public static int add_one(int value) {\n    return value + 1;\n}",
  "checkpoints": [],
  "conflicts": [],
  "assumptions": [
    "Assumed parameter 'value' and return type are standard 32-bit signed integers (int), matching the integer literal '1'.",
    "Preserved primitive addition operator semantics without artificial exception throwing on overflow to maintain 1:1 logic."
  ],
  "helpers_requested": []
}
